# Packages, Namespaces, and Import Forms

**Restart kernel → Run All** for clean execution order.  
This notebook covers what name ends up in your namespace for each `import` form.

In [ ]:
import sys
from pathlib import Path

_demo = Path.cwd() / 'demo_packages'
if not _demo.exists():
    _demo = Path.cwd() / 'modules_and_imports' / 'demo_packages'
assert _demo.exists(), f'demo_packages not found — expected: {_demo}'

if str(_demo) not in sys.path:
    sys.path.insert(0, str(_demo))
print(f'demo_packages on path: {sys.path[0]}')

## 1. Package vs module — the runtime distinction

**At the filesystem level:**
- A module is a `.py` file
- A package is a directory that contains an `__init__.py` file (and optionally submodules)
- A namespace package is a directory **without** `__init__.py` (Python 3.3+)

**At the runtime level:** both are `types.ModuleType` objects.  
The one difference: a **package** has a `__path__` attribute. A plain module does not.

```
__path__ present  →  Python can search inside it for submodules
__path__ absent   →  leaf module, nothing below it
```

In [ ]:
# Watch the print-trace — __init__.py fires as each package is initialized
import pkg_a

print()
has_path = hasattr(pkg_a, '__path__')
print(f'pkg_a.__path__ present: {has_path}  → {list(pkg_a.__path__)}')
print(f'pkg_a.__name__    = {pkg_a.__name__!r}')
print(f'pkg_a.__package__ = {pkg_a.__package__!r}')

In [ ]:
# pkg_a.core is a plain module inside the package
from pkg_a import core

has_path = hasattr(core, '__path__')
print(f'core.__path__ present: {has_path}  (plain module — no submodules)')
print(f'core.__name__    = {core.__name__!r}')
print(f'core.__package__ = {core.__package__!r}')

## 2. __init__.py — the package gateway

When Python imports `pkg_a` (or any submodule of it), `pkg_a/__init__.py` runs **first**.  
This happens exactly once per process, before any submodule is touched.

**What `__init__.py` is for:**
- Mark the directory as a package
- Import names into the package namespace (so users can do `from pkg_a import Counter`)
- Set `__all__` to control what `from pkg_a import *` exposes
- Run any package-level initialization (connection pools, registry setup)

**What to avoid in `__init__.py`:**
- Heavy computation or I/O (it runs at import time, not at call time)
- Importing submodules that aren't always needed (bloats startup time)

In [ ]:
# Import subpackage — watch the init chain: pkg_a.sub.__init__ fires
# pkg_a.__init__ does NOT fire again (already cached)
import pkg_a.sub

print()
print(f'pkg_a.__all__ = {pkg_a.__all__}')
print(f'pkg_a.Counter = {pkg_a.Counter}')   # exposed by __init__.py import
print(f'pkg_a.greet   = {pkg_a.greet}')

In [ ]:
# Full chain: pkg_a.sub.helper — fires helper.py load after sub.__init__
import pkg_a.sub.helper

print()
counter = pkg_a.sub.helper.make_counter(start=5)
print(f'make_counter(5) = {counter}')
counter.increment()
print(f'after increment = {counter}')

## 3. The five import forms — what name lands in namespace

This is one of the most common interview questions. The rule is simple:

> **The name that appears immediately after `import` (or after `as`) is what lands in your namespace.**

```python
import pkg_a              →  'pkg_a'   bound in namespace
import pkg_a.core         →  'pkg_a'   bound (NOT 'core')
from pkg_a import core    →  'core'    bound (NOT 'pkg_a')
from pkg_a.core import Counter  →  'Counter'  bound only
from pkg_a import *       →  whatever __all__ says
```

In [ ]:
# Form 1: import pkg_a
# Only 'pkg_a' lands in your namespace.
# You access submodules via attribute: pkg_a.core.Counter

# (already imported above — pkg_a in sys.modules, no reprint)
import pkg_a

local_names = [n for n in dir() if not n.startswith('_')]
print(f'Names added: pkg_a in locals: {"pkg_a" in local_names}')
print(f'Names added: core  in locals: {"core"  in local_names}')
print()
# Access submodule via attribute chain
print(f'pkg_a.Counter = {pkg_a.Counter}')  # works: exposed by __init__.py

In [ ]:
# Form 2: import pkg_a.core
# 'pkg_a' lands in namespace — NOT 'core' directly.
# BUT Python ensures pkg_a.core is accessible as an attribute.

import pkg_a.core

print(f'pkg_a in locals:      True')
print(f'core  in locals (direct): False  — not bound directly')
print()
print(f'Access via attribute: pkg_a.core.Counter = {pkg_a.core.Counter}')

In [ ]:
# Form 3: from pkg_a import core
# 'core' lands in namespace. 'pkg_a' does NOT.
# sys.modules still has both — the import happened, just no local binding for pkg_a.

from pkg_a import core

print(f'core  in locals: True')
print(f'core.__name__  = {core.__name__!r}')
print(f'Counter via core: {core.Counter}')

In [ ]:
# Form 4: from pkg_a.core import Counter
# Only 'Counter' lands in namespace.
# None of pkg_a, core, or pkg_a.core are bound locally.

from pkg_a.core import Counter, greet

print(f'Counter = {Counter}')
print(f'greet   = {greet}')
print(f'greet("World") = {greet("World")}')

c = Counter(10)
c.increment()
print(f'Counter(10).increment() → {c}')

In [ ]:
# Form 5: from pkg_a import *
# Only imports names listed in __all__.
# If __all__ is absent, imports everything that doesn't start with _.

from pkg_a import *

print(f'pkg_a.__all__ = {pkg_a.__all__}')
print()
# Counter and greet are in __all__ — they land in namespace
print(f'Counter available: {Counter}')
print(f'greet available:   {greet}')

## 4. Relative imports

Relative imports use dots (`.`) to navigate the package hierarchy.

```python
from . import sibling          # from the SAME package
from .sibling import something # from a sibling module
from .. import parent_thing    # one level UP
from ..utils import helper     # sibling of parent
```

**Requirements:**
- The file must be inside a package (i.e., `__package__` is set)
- Relative imports only work when the file is imported as part of a package
- Running a file directly (`python pkg_a/utils.py`) sets `__package__ = None` — relative imports fail

**Absolute vs relative:**

| | Absolute | Relative |
|---|---|---|
| Syntax | `from pkg_a.core import X` | `from .core import X` |
| Works from | anywhere | inside the package only |
| Portable on rename | must update paths | just update the `__init__.py` |
| Interview preference | explicit, always readable | fine for intra-package imports |

In [ ]:
# __package__ tells Python the anchor for relative imports
import pkg_a.sub.helper as helper_mod

print(f'helper_mod.__package__ = {helper_mod.__package__!r}')
print(f'  (the package is pkg_a.sub, so .. means pkg_a)')
print()
# The helper uses: from pkg_a.core import Counter (absolute)
# Equivalent relative: from ..core import Counter
# Both work; absolute is more readable from outside the package

In [ ]:
# Attempting a relative import from a top-level script raises ImportError
# This simulates what happens if you run: python pkg_a/utils.py

try:
    import importlib
    # Import utils as a top-level module (not as part of pkg_a)
    # This breaks relative imports because __package__ is '' or None
    import importlib.util
    spec = importlib.util.spec_from_file_location(
        '__main__',           # pretend it's run as a script
        str(_demo / 'pkg_a' / 'utils.py')
    )
    mod = importlib.util.module_from_spec(spec)
    # Would fail if utils.py used relative imports
    # Our utils.py uses absolute imports so it won't fail — but the principle holds
    print('utils.py uses absolute imports — works fine')
    print('If it used "from .core import greet", this would raise ImportError')
except Exception as e:
    print(f'{type(e).__name__}: {e}')

## 5. __all__ and the public API

`__all__` is a list of strings that controls what `from module import *` exports.
It also serves as documentation: "these are the public names of this module."

**Without `__all__`:** `import *` imports everything that doesn't start with `_`.  
**With `__all__`:** `import *` imports exactly the listed names and nothing else.

`__all__` does **not** prevent explicit imports: you can always do  
`from pkg_a._private import secret` even if `_private` isn't in `__all__`.

**Convention:** define `__all__` in any module that's part of a public API.

In [ ]:
import pkg_a

print(f'pkg_a.__all__ = {pkg_a.__all__!r}')
print()
# Names exposed at pkg_a level (via __init__.py imports)
public = [n for n in dir(pkg_a) if not n.startswith('_')]
print(f'dir(pkg_a) non-private names: {public}')

## 6. Namespace packages (Python 3.3+)

A directory without `__init__.py` is still importable as a **namespace package**.  
Python's `PathFinder` creates a namespace package with `__path__` but no `__file__`.

**Use case:** split a single logical package across multiple directories (e.g., plugin systems,  
split source trees in monorepos). Django and large frameworks sometimes use this.

**Key difference from regular packages:**

| | Regular package | Namespace package |
|---|---|---|
| `__init__.py` | Required | Absent |
| `__file__` | Points to `__init__.py` | `None` |
| `__path__` | One directory | Can span multiple dirs |
| Initialization code | In `__init__.py` | Nowhere — just a container |

In [ ]:
# Quick comparison: regular package vs namespace package attributes
import pkg_a   # regular package
import email   # stdlib namespace package

print('pkg_a (regular package):')
print(f'  __file__: {pkg_a.__file__!r}')
print(f'  __path__: {list(pkg_a.__path__)}')
print()
print('email (regular package with __init__.py):')
has_file = hasattr(email, '__file__') and email.__file__ is not None
print(f'  has __file__: {has_file}')
print(f'  __path__: {list(email.__path__)}')